## prepare


In [1]:
import numpy as np
np.float_ = np.float64
import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch.utils.data import TensorDataset
import datasets
from datasets import load_from_disk, Image
import torch
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from torchmetrics.image.fid import FrechetInceptionDistance
import math
import copy

In [2]:
import os
import random
import numpy as np
import torch


def set_seed(seed: int):
    # 1. Python random
    random.seed(seed)

    # 2. NumPy
    np.random.seed(seed)

    # 3. PyTorch (CPU & GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果你用多个 GPU

    # 4. cuDNN 设置为确定性模式，关闭 benchmark
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True





def seed_worker(worker_id):
    # 每个 worker 基于相同或偏移后的种子
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)


# 在脚本最上方调用一次
SEED = 42
set_seed(SEED)

# PyTorch 1.7+ 可直接用 generator
gen = torch.Generator()
gen.manual_seed(SEED)

### dataset

In [3]:
cuda_num = 2
s = 32
p = 3
INPUT_DIM = p * s * s
LATENT_DIM = 20

HIDDEN_DIM = LATENT_DIM * 10
data_folder = "data_cifar10"
datasetname = "cif10"

# train+label
train_dataset = datasets.load_from_disk(f"/home/yzm/{data_folder}/train")
print(f"Total samples: {len(train_dataset)}")

train_images = (
    torch.stack([torch.tensor(np.array(img)) for img in train_dataset["img"]]).float()
    / 255.0
)
# CIFAR10: [N, 32, 32, 3] → 转成 [N, 3, 32, 32]
train_images = train_images.permute(0, 3, 1, 2)
train_images = train_images.flatten(start_dim=1)  # [N, 3072]

train_labels = torch.tensor(train_dataset["label"], dtype=torch.long)
train_dataset_torch = TensorDataset(train_images, train_labels)


# test+label
test_dataset = datasets.load_from_disk(f"/home/yzm/{data_folder}/test")
print(f"Total samples: {len(test_dataset)}")

test_images = (
    torch.stack([torch.tensor(np.array(img)) for img in test_dataset["img"]]).float()
    / 255.0
)
test_images = test_images.permute(0, 3, 1, 2)
test_images = test_images.flatten(start_dim=1)   # [N, 3072]

test_labels = torch.tensor(test_dataset["label"], dtype=torch.long)
test_dataset_torch = TensorDataset(test_images, test_labels)

Total samples: 50000
Total samples: 10000


In [4]:
from torch.utils.data import random_split

# 定义划分比例
train_size = int(0.9 * len(train_dataset_torch))  # 90%
val_size = len(train_dataset_torch) - train_size  # 10%

# 按比例划分数据集
train_dataset_torch, val_dataset_torch = random_split(train_dataset_torch, [train_size, val_size], generator=gen)

# 打印划分后的数据集大小
print(f"Train dataset size: {len(train_dataset_torch)}")
print(f"Validation dataset size: {len(val_dataset_torch)}")

Train dataset size: 45000
Validation dataset size: 5000


### vae class


In [5]:
class VAE_Encoder(nn.Module):
    """
    VAE的encoder，将输入的图片映射到隐变量的分布，输出隐变量分布的均值和方差.
    由于方差不能为负，所以这里输出的是log(方差)，而不是方差本身.
    """

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE_Encoder, self).__init__()

        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.Tanh())

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)


class VAE_Decoder(nn.Module):
    """
    VAE的decoder，将隐变量映射回图片.
    """

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE_Decoder, self).__init__()

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.decoder(z)


class VAE(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM):
        super(VAE, self).__init__()

        self.latent_dim = latent_dim

        self.encoder = VAE_Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = VAE_Decoder(input_dim, hidden_dim, latent_dim)

    def encode(self, x):
        """
        将输入的图片映射到隐变量的分布，输出隐变量分布的均值和log(方差).
        """
        return self.encoder(x)

    def decode(self, z):
        """
        将隐变量映射回图片.
        """
        return self.decoder(z)

    def reparameterize(self, mu, logvar):
        """
        重参数化技巧，先从标准正态分布中采样一个epsilon，然后根据隐变量分布的均值和方差，计算出隐变量.
        """
        std = torch.exp(
            0.5 * logvar
        )  # 计算标准差, std = sqrt(var) = sqrt(exp(logvar)) = exp(logvar/2)
        epsilon = torch.randn_like(
            std, requires_grad=False
        )  # 从标准正态分布中采样epsilon
        return mu + epsilon * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

### loss


In [6]:
def loss_iwae_gauss(mu, logvar, x, decoder, k=5, sigma2=1.0, device=None):
    if device is None:
        device = x.device

    B = x.size(0)
    D = x.view(B, -1).size(1)
    z_dim = mu.size(-1)

    # 采样 k 个 z: shape [B, k, Z]
    eps = torch.randn(B, k, z_dim, device=device)
    std = torch.exp(0.5 * logvar)                          # [B, Z]
    z = mu.unsqueeze(1) + std.unsqueeze(1) * eps           # [B, k, Z]

    # 常量
    dtype = z.dtype
    log_2pi = torch.tensor(math.log(2 * math.pi), device=device, dtype=dtype)
    log_k = torch.tensor(math.log(k), device=device, dtype=dtype)

    # 第二项 log p(z) 标准正态先验: [B, k]
    log_pz = -0.5 * (z**2 + log_2pi).sum(-1)

    # 第三项 log q(z|x): [B, k]
    log_qz = -0.5 * (
        ((z - mu.unsqueeze(1)) ** 2) / (std.unsqueeze(1) ** 2)
        + logvar.unsqueeze(1)
        + log_2pi
    ).sum(-1)

    # 解码: flatten z -> [B*k, Z] and decode -> [B*k, D]
    z_flat = z.view(B * k, z_dim)
    x_recon = decoder(z_flat)                                # [B*k, D]
    x_expand = x.unsqueeze(1).expand(-1, k, -1).contiguous().view(B * k, -1)
    diff = x_expand - x_recon                                # [B*k, D]

    # 第一项 log p(x|z) 似然，固定方差 sigma2: [B, k]
    const_px = torch.tensor(0.5 * D * math.log(2 * math.pi * sigma2), device=device, dtype=diff.dtype)
    log_px = (-0.5 * (diff**2 / sigma2).sum(-1) - const_px).view(B, k)

    # importance weights: log-sum-exp 聚合 （per data point: [B]）
    log_w = log_px + log_pz - log_qz                        # [B, k]
    m = log_w.max(dim=1, keepdim=True).values               # [B, 1]
    sum_exp = torch.exp(log_w - m).sum(dim=1)               # [B]
    Lk = m.squeeze(1) + torch.log(sum_exp) - log_k          # [B]

    total_loss = - Lk.sum()
    return total_loss


In [7]:
def loss_function(x, recon_x, mu, logvar, prior_var=1.0, sigma2=1.0):
    """
    修改版: 支持动态调整先验方差 prior_var
    prior_var: scalar, 先验分布的方差 (默认1.0)
    """

    # 2. KL 散度 (General Gaussian Prior)
    # log(prior_var)
    log_prior_var = math.log(prior_var)
    
    # 公式推导转代码:
    # 0.5 * sum(log(sigma_p^2) - log(sigma^2) - 1 + (sigma^2 + mu^2)/sigma_p^2)
    # 提取 -0.5: -0.5 * sum(1 + log(sigma^2) - log(sigma_p^2) - (sigma^2 + mu^2)/sigma_p^2)
    
    var = logvar.exp()
    KLD = -0.5 * torch.sum(1 + logvar - log_prior_var - (mu.pow(2) + var) / prior_var)
    
    return KLD

In [8]:
def score_fn_torch(z: torch.Tensor, x_target: torch.Tensor, decoder: nn.Module, sigma2=1.0, prior_var=1.0) -> torch.Tensor:
    """
    修改版: 引入 prior_var 参数
    计算 ∇_z log p(z, x_target)
    """
    assert z.requires_grad, "z must have requires_grad=True"

    B, D = z.shape[0], z.shape[1]
    if x_target.ndim == 1:
        x_target = x_target.unsqueeze(0)
    
    x_target = x_target.view(B, -1)   # [B, 3072]
    x_rec = decoder(z).view(B, -1)
    diff = x_target - x_rec

    # --- 修改点 1: log p(z) 考虑先验方差 ---
    # log p(z) = -0.5 * (z^2 / prior_var)
    logpz = -0.5 * (z ** 2 / prior_var).sum(dim=1)

    # log p(x|z) = -1/(2σ²) * ||x - decoder(z)||²
    logpx = -0.5 / sigma2 * (diff ** 2).sum(dim=1)

    # --- 分别求两项梯度 ---
    # --- 修改点 2: grad_pz 除以先验方差 ---
    grad_pz = -z / prior_var  # [B, D]
    
    grad_px = torch.autograd.grad(logpx.sum(), z, create_graph=True, retain_graph=True)[0]

    # 合并两项得到 g(z)
    g = grad_pz + grad_px  # [B, D]
    
    return g

In [9]:
def bamloss_batch(
    mu, logvar, x_target, decoder,
    sample_K=2, lambda_t=0.1, sigma2=1.0, prior_var=1.0, # <--- 新增参数
):

    N, D = mu.shape

    # === 1. 重参数化采样 z (这就是你报错缺失的部分) ===
    std = torch.exp(0.5 * logvar)
    var = std**2
    # 采样 epsilon
    eps = torch.randn(N, sample_K, D, device=mu.device)
    # 得到 z: [N, K, D]
    z = mu.unsqueeze(1) + eps * std.unsqueeze(1)
    # 必须开启梯度，因为后面要算 score
    z.requires_grad_(True)

    # === 2. 计算 score ===
    z_flat = z.view(N * sample_K, D)
    x_rep = x_target.unsqueeze(1)\
                     .expand(N, sample_K, *x_target.shape[1:])\
                     .reshape(N * sample_K, *x_target.shape[1:])
    
    # 调用修改后的 score_fn_torch，传入 prior_var
    scores_flat = score_fn_torch(z_flat, x_rep, decoder, sigma2=sigma2, prior_var=prior_var)
    scores = scores_flat.view(N, sample_K, D)

    # === 计算 Loss 项 ===
    # term1: 因为 prior_var = 1, 所以是 D / 1.0 = D
    
    
    # term2 和 term3 的计算保持原样，它们内部使用的 scores 已经包含了 sigma2 的影响
    g_norm2_sigma = (var.unsqueeze(1) * scores**2).sum(dim=2)
    align_term = ((z - mu.unsqueeze(1)) * scores).sum(dim=2)
    
    term1 = D / prior_var
    term2 = g_norm2_sigma.mean(dim=1)
    term3 = 2 * align_term.mean(dim=1) 

    # 如果你想做敏感性测试，记得加上 term1
    score_div =  term2 + term3
    
    loss = score_div.mean()  # 对 batch 求平均

    return loss

In [10]:
def cv_bamloss_batch(
    mu, logvar, x_target, decoder,
    sample_K=2, lambda_t=0.1, sigma2=0.1,
    writer=None, global_step=None,
):
    """
    向量化的 BaM 损失，支持 batch 并行 (集成 Control Variates 版本)。
    """
    N, D = mu.shape
    eps = 1e-8 # 防止除0

    # -------------------------------------------------------------------------
    # [Control Variates 准备阶段] 计算 g(mu) 和 grad(f1(mu))
    # -------------------------------------------------------------------------
    # 确保 mu 计算梯度
    mu.requires_grad_(True)
    
    # 1. 计算 g(mu) [N, D]
    # 注意：这里直接使用 x_target，因为 mu 对应每个 batch 只有 1 个
    g_mu = score_fn_torch(mu, x_target, decoder, sigma2)

    # 2. 计算 f1(mu) = g(mu)^T Sigma g(mu)
    std = torch.exp(0.5 * logvar)
    var = std**2
    f1_mu = (var * g_mu**2).sum(dim=1) # [N]

    # 3. 计算 grad(f1(mu)) [N, D]
    grad_f1_mu = torch.autograd.grad(
        outputs=f1_mu.sum(), inputs=mu, create_graph=True
    )[0]

    # -------------------------------------------------------------------------
    # [原有采样逻辑]
    # -------------------------------------------------------------------------
    # 1. 重参数化采样 z: [N, K, D]
    eps_z = torch.randn(N, sample_K, D, device=mu.device)
    z = mu.unsqueeze(1) + eps_z * std.unsqueeze(1)
    z.requires_grad_(True)

    # 2. 计算 score，先 reshape 到 [N*K, D]
    z_flat = z.view(N * sample_K, D)
    x_rep = x_target.unsqueeze(1)               \
                    .expand(N, sample_K, *x_target.shape[1:]) \
                    .reshape(N * sample_K, *x_target.shape[1:])
    scores_flat = score_fn_torch(z_flat, x_rep, decoder, sigma2)
    scores = scores_flat.view(N, sample_K, D)

    # -------------------------------------------------------------------------
    # [Control Variates 计算阶段]
    # -------------------------------------------------------------------------
    z_minus_mu = z - mu.unsqueeze(1) # [N, K, D]

    # 原始项 (Raw Terms) [N, K]
    # Term 2 raw: ||g_l||_Σ^2
    g_norm2_sigma = (var.unsqueeze(1) * scores**2).sum(dim=2) 
    # Term 3 raw (不含系数2): (z_l - μ)^T * g_l
    align_term = (z_minus_mu * scores).sum(dim=2)

    # 控制变量项 (CV Terms) [N, K]
    # CV1: grad(f1(mu))^T (z - mu)
    cv1_vals = (grad_f1_mu.unsqueeze(1) * z_minus_mu).sum(dim=2)
    # CV2: g(mu)^T (z - mu)
    cv2_vals = (g_mu.unsqueeze(1) * z_minus_mu).sum(dim=2)

    # 计算最优系数 c1, c2 (对 K 维度求统计量) [N]
    # c = Cov(Term, CV) / Var(CV)
    def compute_optimal_c(term_vals, cv_vals):
        # 1. 中心化
        term_centered = term_vals - term_vals.mean(dim=1, keepdim=True)
        cv_centered = cv_vals - cv_vals.mean(dim=1, keepdim=True)
        
        # 2. 分子：Covariance
        numerator = (term_centered * cv_centered).mean(dim=1)
        
        # 3. 分母：Variance
        # 【修改点1】加大 eps，从 1e-8 加到 1e-4 或 1e-5，避免 K=2 时方差过小导致的除零爆炸
        denominator = cv_vals.var(dim=1, unbiased=False) + 1e-5 
        
        c = numerator / denominator
        
        # 【修改点2】Detach！我们只希望 c 作为降低方差的系数，不希望优化器去通过改变 c 来优化 Loss
        c = c.detach()
        
        # 【修改点3】截断 (Clamping)！
        # 如果 c 计算出来是几千几万，说明 CV 和原始项的相关性出了问题或者方差估计完全错误。
        # 限制在 [-100, 100] 或更小的范围内，保证训练初期 Loss 不崩。
        #c = torch.clamp(c, min=-100.0, max=100.0)
        
        return c

    c1 = compute_optimal_c(g_norm2_sigma, cv1_vals)
    c2 = compute_optimal_c(align_term, cv2_vals)

    # -------------------------------------------------------------------------
    # [Score-matching 三项 (应用 CV)]
    # -------------------------------------------------------------------------
    
    # 1) tr(I) = D
    term1 = D
    
    # 2) Term 2: Mean(Raw - c1 * CV)
    # 1/L ∑ ||g_l||_Σ^2 - c1 * 1/L ∑ CV1
    term2_corrected = g_norm2_sigma - c1.unsqueeze(1) * cv1_vals
    term2 = term2_corrected.mean(dim=1) # [N]
    
    # 3) Term 3: Mean(Raw - c2 * CV) * 2
    # 2/L ∑ (z_l - mu)^T g_l - 2 * c2 * 1/L ∑ CV2
    term3_corrected = align_term - c2.unsqueeze(1) * cv2_vals
    term3 = 2 * term3_corrected.mean(dim=1) # [N]

    # 汇总 Loss
    score_div = term2 + term3

    # 6. 最终 loss 对 batch 求平均
    loss = score_div.mean() # 对 batch 求平均

    return loss

### eval

In [11]:
# # grayscale version (for MNIST)
# def eval_fid(model, test_loader, device, fid_metric):
#     model.eval()
#     fid_metric.reset()
#
#     with torch.no_grad():
#         for real, _ in test_loader:
#             real = real.to(device)
#
#             # 编码—采样—解码
#             mu, logvar = model.encode(real)
#             z = model.reparameterize(mu, logvar)
#             fake = model.decode(z)
#
#             # 从 [B, D] 恢复成 [B,1,side,side]
#             B, D = real.shape
#             side = int(D**0.5)
#             real_img = real.view(B, 1, side, side).repeat(1, 3, 1, 1)
#             fake_img = fake.view(B, 1, side, side).repeat(1, 3, 1, 1)
#
#             # resize 到 299×299
#             real_rs = F.interpolate(
#                 real_img, size=(299, 299), mode="bilinear", align_corners=False
#             )
#             fake_rs = F.interpolate(
#                 fake_img, size=(299, 299), mode="bilinear", align_corners=False
#             )
#
#             # 更新 FID 统计（uint8）
#             fid_metric.update((real_rs * 255).to(torch.uint8), real=True)
#             fid_metric.update((fake_rs * 255).to(torch.uint8), real=False)
#
#     # 计算并返回得分
#     fid_score = fid_metric.compute().item()
#     model.train()
#     return fid_score

In [12]:
def eval_fid(model, test_loader, device, fid_metric):
    model.eval()
    fid_metric.reset()

    with torch.no_grad():
        for real, _ in test_loader:
            real = real.to(device)

            # 编码—采样—解码
            mu, logvar = model.encode(real)
            z = model.reparameterize(mu, logvar)
            fake = model.decode(z)

            # 从 [B, D] 恢复成 [B,3,s,s]
            B, D = real.shape
            real_img = real.view(B, 3, s, s)
            fake_img = fake.view(B, 3, s, s)

            # resize 到 299×299
            real_rs = nn.functional.interpolate(real_img, size=(299, 299), mode="bilinear", align_corners=False)
            fake_rs = nn.functional.interpolate(fake_img, size=(299, 299), mode="bilinear", align_corners=False)

            # 更新 FID 统计（uint8）
            fid_metric.update((real_rs * 255).to(torch.uint8), real=True)
            fid_metric.update((fake_rs * 255).to(torch.uint8), real=False)

    # 计算并返回得分
    fid_score = fid_metric.compute().item()
    model.train()
    return fid_score

In [13]:
import math
@torch.no_grad()
def evaluate_iwae_gauss(model, dataloader, device, k=5000, sigma2=1.0):
    total_loglik = 0.0
    total_samples = 0
    D = None

    for x, _ in dataloader:
        x = x.to(device)
        B, D = x.shape
        total_samples += B

        # 编码
        mu, logvar = model.encode(x)  # [B, Z]
        std = torch.exp(0.5 * logvar)  # [B, Z]

        # 采样 k 个 z
        eps = torch.randn(B, k, mu.size(1), device=device)
        z = mu.unsqueeze(1) + std.unsqueeze(1) * eps  # [B, k, Z]

        # 第二项：log p(z) 标准正态先验
        log_pz = -0.5 * (z**2 + math.log(2 * math.pi)).sum(-1)  # [B, k]

        # 第三项：log q(z|x)
        log_qz = -0.5 * (
            ((z - mu.unsqueeze(1)) ** 2) / (std.unsqueeze(1) ** 2)
            + logvar.unsqueeze(1)
            + math.log(2 * math.pi)
        ).sum(
            -1
        )  # [B, k]

        # 解码
        x_recon = model.decode(z.view(B * k, -1))  # [B*k, D]
        x_expand = x.unsqueeze(1).repeat(1, k, 1).view(B * k, -1)
        diff = x_expand - x_recon  # [B*k, D]

        # 第一项：log p(x|z) 似然，固定方差 sigma2
        log_px = (
            -0.5 * (diff**2 / sigma2).sum(-1) - 0.5 * D * math.log(2 * math.pi * sigma2)
        ).view(
            B, k
        )  # [B, k]

        # importance weights: log-sum-exp 聚合
        log_w = log_px + log_pz - log_qz  # [B, k]
        m = log_w.max(dim=1, keepdim=True).values  # [B, 1]
        Lk = (
            m.squeeze(1) + torch.log(torch.exp(log_w - m).sum(dim=1)) - math.log(k)
        )  # [B]

        total_loglik += Lk.sum().item()

    avg_Lk = total_loglik / total_samples
    avg_NLL = -avg_Lk
    bpd = avg_NLL / (D * math.log(2))  # bits/dim
    return avg_Lk, avg_NLL, bpd

### plt

In [14]:
# # grayscale version (for MNIST)
# def write_test_images(writer, model, data_loader, device, global_step, sample_K=7,folder_name="img"):
#     model.eval()
#     batch, _ = next(iter(data_loader))
#     batch = batch.to(device)
#     K = min(sample_K, batch.size(0))
#     inputs = batch[:K].cpu().view(K, 28, 28)
#
#     with torch.no_grad():
#         mu, logvar = model.encode(batch)
#         sigma = torch.exp(0.5 * logvar)
#         eps = torch.randn_like(sigma)
#         z = mu + eps * sigma
#         recon = model.decode(z[:K]).cpu().view(K, 28, 28).numpy()
#
#     fig = plt.figure(figsize=(15, 6))
#     for i in range(K):
#         ax = fig.add_subplot(2, K, i + 1)
#         ax.imshow(inputs[i], cmap="gray")
#         ax.axis("off")
#         ax = fig.add_subplot(2, K, K + i + 1)
#         ax.imshow(recon[i], cmap="gray")
#         ax.axis("off")
#
#     writer.add_figure(f"{folder_name}/{global_step}-rec", fig, global_step)
#     plt.close(fig)

In [15]:
# RGB
def write_test_images(
    writer,
    model,
    data_loader,
    device,
    global_step,
    sample_K=7,
    img_size=(s, s),
    num_channels=3,
    folder_name="img",
):
    """
    在 TensorBoard 上展示 CIFAR-10 原图 & VAE 重构图。
    """
    model.eval()

    # 1) 从 data_loader 累积扁平张量 flat_imgs 直到 >= sample_K
    all_flat = []
    for flat_img, _ in data_loader:
        all_flat.append(flat_img)
        if sum(x.size(0) for x in all_flat) >= sample_K:
            break
    flat_imgs = torch.cat(all_flat, dim=0).to(device)

    B = flat_imgs.size(0)
    K = min(sample_K, B)
    C, H, W = num_channels, *img_size

    # 2) 把前 K 张先恢复成 [K, C, H, W] 供可视化
    inputs = flat_imgs[:K].view(K, C, H, W).cpu()

    # 3) 编码 → 样本 → 解码
    with torch.no_grad():
        mu, logvar = model.encode(flat_imgs)
        sigma = torch.exp(0.5 * logvar)
        eps = torch.randn_like(sigma)
        z = mu + eps * sigma

        recon_flat = model.decode(z[:K]).cpu()
        recon = recon_flat.view(K, C, H, W)

    # 4) 转成 HWC NumPy 数组
    inputs_np = inputs.permute(0, 2, 3, 1).contiguous().numpy()
    recon_np = recon.permute(0, 2, 3, 1).contiguous().numpy()

    # 5) 按组绘图，每组最多 7 张
    group_size = 7
    for group_idx in range(0, K, group_size):
        end_idx = min(group_idx + group_size, K)
        current_K = end_idx - group_idx

        fig = plt.figure(figsize=(2 * current_K, 4))
        for i in range(current_K):
            ax1 = fig.add_subplot(2, current_K, i + 1)
            ax1.imshow(inputs_np[group_idx + i])
            ax1.axis("off")

            ax2 = fig.add_subplot(2, current_K, current_K + i + 1)
            ax2.imshow(recon_np[group_idx + i])
            ax2.axis("off")

        writer.add_figure(
            f"{folder_name}/{global_step}-{group_idx // group_size}",
            fig,
            global_step,
        )
        plt.close(fig)
    model.train()

In [16]:
def log_latent_space(writer, model, dataloader, global_step, device):
    model.eval()
    all_mu = []
    all_logvar = []
    all_labels = []
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            mu, logvar = model.encode(x)
            all_mu.append(mu.cpu())
            all_logvar.append(logvar.cpu())
            all_labels.extend(y.cpu().tolist())

    all_mu = torch.cat(all_mu, dim=0)  # [N, z_dim]
    all_logvar = torch.cat(all_logvar, dim=0)  # [N, z_dim]
    all_labels = [str(l) for l in all_labels]  # 转成字符串，方便 Color by

    max_points = 2000
    all_mu = all_mu[:max_points]
    all_labels = all_labels[:max_points]
    all_logvar = all_logvar[:max_points]

    # print("mu 是否有nan: ", torch.isnan(all_mu).any(), torch.isinf(all_mu).any())
    # print("logvar 是否有nan: ", torch.isnan(all_logvar).any(), torch.isinf(all_logvar).any())
    # print("数据维度: ", all_mu.shape, all_logvar.shape, len(all_labels))
    # print("数据范围: ", all_mu.min().item(), all_mu.max().item(), all_logvar.min().item(), all_logvar.max().item())
    # print("数据方差: ", all_mu.std(dim=0), all_logvar.std(dim=0))

    # print("标签类别: ", set(all_labels))
    # 写入 TensorBoard Embedding
    writer.add_embedding(
        mat=all_mu,
        metadata=all_labels,  # 标签（比如数字类别）
        tag="latent_mu",
        global_step=global_step,
    )
    writer.add_embedding(
        mat=all_logvar,
        metadata=all_labels,  # 标签（比如数字类别）
        tag="latent_logvar",
        global_step=global_step,
    )

In [17]:
def log_latent_stats(model, loader, writer, epoch, device):
    model.eval()
    ############## 写 mu/std/z m##############
    all_mu, all_std, all_z = [], [], []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            mu, logvar = model.encode(x)  # [B, Z]
            std = torch.exp(0.5 * logvar)  # [B, Z]
            # 如果你想按重参数化采样的 z
            eps = torch.randn_like(std)
            z = mu + eps * std  # [B, Z]

            all_mu.append(mu.cpu())
            all_std.append(std.cpu())
            all_z.append(z.cpu())

    # 拼接成 [N, Z]
    all_mu = torch.cat(all_mu, dim=0)
    all_std = torch.cat(all_std, dim=0)
    all_z = torch.cat(all_z, dim=0)

    # 对样本维度取均值 → [Z]
    mu_mean = all_mu.mean(dim=0)  # Tensor of shape [Z]
    std_mean = all_std.mean(dim=0)
    z_mean = all_z.mean(dim=0)

    # 控制台打印（可选）
    print(f"Epoch {epoch+1}  mu_mean per dim:  {mu_mean.tolist()}")
    print(f"Epoch {epoch+1}  std_mean per dim: {std_mean.tolist()}")
    print(f"Epoch {epoch+1}  z_mean per dim:   {z_mean.tolist()}")

    # 写入 TensorBoard：每个潜变量维度一个 scalar
    for i in range(mu_mean.size(0)):
        writer.add_scalar(f"latent/mu_dim{i}", mu_mean[i].item(), epoch)
        writer.add_scalar(f"latent/std_dim{i}", std_mean[i].item(), epoch)
        writer.add_scalar(f"latent/z_dim{i}", z_mean[i].item(), epoch)

    # 如果还想看分布，可写 histogram：
    writer.add_histogram("histogram/mu", all_mu, epoch)
    writer.add_histogram("histogram/std", all_std, epoch)
    writer.add_histogram("histogram/z", all_z, epoch)

    ############## 写 g_dim ##############
    Z = all_mu.size(1)  # 潜变量维度 5

    for d in range(Z):
        vals = epoch_metrics_g[f"g_abs_dim{d}"]  # list of length num_batches
        avg = sum(vals) / len(vals)  # 维度 d 上的平均梯度
        writer.add_scalar(f"latent_g/g_abs_dim{d}", avg, epoch)

        vals = epoch_metrics_g[f"g_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g/g_dim{d}", avg, epoch)

    # 写 g 的整体分布直方图
    all_g = []
    for d in range(Z):
        all_g.extend(epoch_metrics_g[f"g_dim{d}"])  # 收集所有维度的 g

    all_g_tensor = torch.tensor(all_g, dtype=torch.float32)
    writer.add_histogram("histogram_g/g", all_g_tensor, epoch)

    print(f"Epoch {epoch+1}  g per dim: {[epoch_metrics_g[f'g_abs_dim{d}'][-1] for d in range(Z)]}")

    ############### 写 g 的分解项 ##############
    for d in range(Z):
        vals = epoch_metrics_g[f"g_pz_abs_dim{d}"]  # list of length num_batches
        avg = sum(vals) / len(vals)  # 维度 d 上的平均
        writer.add_scalar(f"latent_g_pz/g_pz_abs_dim{d}", avg, epoch)
        vals = epoch_metrics_g[f"g_px_abs_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g_px/g_px_abs_dim{d}", avg, epoch)

        vals = epoch_metrics_g[f"g_pz_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g_pz/g_pz_dim{d}", avg, epoch)
        vals = epoch_metrics_g[f"g_px_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g_px/g_px_dim{d}", avg, epoch)

    # g_pz 和 g_px 的整体分布直方图
    all_g_pz = []
    all_g_px = []
    for d in range(Z):            
        all_g_pz.extend(epoch_metrics_g[f"g_pz_dim{d}"])
        all_g_px.extend(epoch_metrics_g[f"g_px_dim{d}"])

    all_g_pz_tensor = torch.tensor(all_g_pz, dtype=torch.float32)
    all_g_px_tensor = torch.tensor(all_g_px, dtype=torch.float32)
    writer.add_histogram("histogram_g/g_pz", all_g_pz_tensor, epoch)
    writer.add_histogram("histogram_g/g_px", all_g_px_tensor, epoch)

    print(f"Epoch {epoch+1}  g_pz per dim: {[epoch_metrics_g[f'g_pz_abs_dim{d}'][-1] for d in range(Z)]}")
    print(f"Epoch {epoch+1}  g_px per dim: {[epoch_metrics_g[f'g_px_abs_dim{d}'][-1] for d in range(Z)]}")

    model.train()  # 别忘了恢复训练模式

## run cif10

In [18]:
#### ours
batch_size0 = 128
batch_size_test = 64
batch_size_test_nll = 16
epoch0 = 100
lr0 = 1e-4


sigma20 = 1
prior_var0 = 1
lambda_t0 = 0.1
lambda_KLt = 0.1
lambda_D0 = 0.5
lambda_KL0 = 0
lambda_sum = 0.5 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1}
lambda_KL0 = 0 # vae的 KL，非vae时=0

torch.cuda.empty_cache()

device = torch.device(f"cuda:{cuda_num}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

flag_vae = False
flag_iwae = False
flag_cv = False
prefix = f"F3-add-latent{LATENT_DIM}"
folder = "runs__re"
file_idx = 4444
sample_K0 = 2

lambda_D0 = 0.5
lambda_sum = 1 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1} 的分母
num_KLt = 1.0 # BaM 的 KL{t+1} 的分子

sigma20 = 1
prior_var0 = 1
sigma2_list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
prior_var_list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
sigma2_list = [0.01, 1.0, 10.0]
prior_var_list = [0.01, 1.0, 10.0]
for lr0 in [5e-4]:
    for sigma20 in [1]:
        for prior_var0 in prior_var_list:    
            for tensround in range(file_idx, file_idx + 1):
                time_now = datetime.now().strftime("%m-%d_%H-%M")
                if flag_vae:
                    lambda_sum = 0
                    lambda_KLt = 1
                    lambda_KL0 = 1
                    model_dir=f"{folder}/{prefix}-{datasetname}-vae-s2lk{sigma20}-s2p{prior_var0}-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir 
                    )
                elif flag_iwae:
                    model_dir=f"{folder}/{prefix}-{datasetname}-iwae-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                elif flag_cv:
                    model_dir=f"{folder}/{prefix}-{datasetname}-cv-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                else:
                    model_dir=f"{folder}/{prefix}-{datasetname}-ours-s2lk{sigma20}-s2p{prior_var0}-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{num_KLt/lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                    log_dir=model_dir
                )
                model = VAE().to(device)
                model_t = VAE().to(device)
                optimizer = torch.optim.Adam(model.parameters(), lr=lr0)

                fid_metric = FrechetInceptionDistance(feature=2048).to(device)

                data_loader = DataLoader(
                    train_dataset_torch, batch_size=batch_size0, shuffle=True,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                val_loader = DataLoader(
                    val_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4, worker_init_fn=seed_worker, generator=gen
                )
                                

                test_loader = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                test_loader_nll = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test_nll, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )

                plt_ck_p = epoch0 // 20  # 显示 20 次测试结果

                total_steps = epoch0 * len(data_loader)

                for epoch in range(epoch0):

                    losses = []
                    Dqs = []
                    KLs = []
                    KL_ts = []
                    MSEs = []
                    BCEs = []
                    sum_Dq_plus_lKLt_dot_KLt_s = []

                    alpha_max = 1.0
                    beta_max = 1.0
                    gamma_max = 1.0

                    warmup1 = 30      # 只用 MSE+KL
                    warmup2 = 80      # 完整过渡到 MSE+KL+BaM
                    # alpha = alpha_max * min(1.0, epoch / warmup1)
                    alpha = alpha_max
                    gamma = gamma_max
                    beta = beta_max
                    
                    old_enc_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.encoder.state_dict().items()
                    }

                    for batch_idx, (x, y) in enumerate(data_loader):
                        x = x.to(device)  # [batch, ...]
                        recon_batch, mu, logvar = model(x)  # [batch, D]

                        # 恢复旧 encoder 的参数到临时 old_encoder 推断（在 CPU 拷贝上不需要梯度）
                        with torch.no_grad():
                            # 将保存的 old state 恢复到 model.encoder 的一个临时副本计算 mu_old/logvar_old
                            # 更高效：直接在当前 encoder 上 load_state_dict(old_enc_state)，但要小心线程安全与 GPU；这里用简单方式：
                            # 创建新 encoder 实例会更健壮但也更重 —— 下面采用直接把旧参数载入 model.encoder 并再恢复当前参数的办法（较快）
                            model_t.encoder.load_state_dict(old_enc_state)   # load old weights
                            # 注意：mu_old, logvar_old 都是在 no_grad 环境下计算的（无需保持 grad）
                            mu_old, logvar_old = model_t.encoder(x)         # compute old outputs

                        global_step = epoch * len(data_loader) + batch_idx

                        KL = 1.0 / x.size(0) * loss_function(x, recon_batch, mu, logvar, sigma2=sigma20, prior_var=prior_var0)
                        KLs.append(KL.item())
                        
                        if not flag_vae and not flag_iwae:                            
                            if flag_cv:
                                Dq = cv_bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K1=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    writer=writer, global_step=global_step)
                            else:
                                Dq = bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    prior_var=prior_var0)
                        else:
                            Dq = torch.tensor(0.0)
                        Dqs.append(Dq.item())
                        
                        MSE = nn.functional.mse_loss(
                            recon_batch, x, reduction="sum"
                        ) / x.size(0)

                        MSEs.append(MSE.item())
                        MSE = (1.0 / sigma20) * MSE

                        var = torch.exp(logvar)
                        var_old = torch.exp(logvar_old)
                        # 先在 latent 维度 D 上求和
                        KL_t_per_latent = 0.5 * (
                            torch.log(var / var_old) +
                            (var_old + (mu_old - mu).pow(2)) / var - 1
                        ).sum(dim=1)                        # [N]
                        # 再对 batch 维度求均值
                        KL_t = KL_t_per_latent.mean()        # scalar

                        KL_ts.append(KL_t.item())

                        Dq *= lambda_D0
                        KL *= lambda_KL0
                        KL_t *= num_KLt / lambda_KLt

                        sum_Dq_plus_lKLt_dot_KLt = Dq + KL_t
                        sum_Dq_plus_lKLt_dot_KLt_s.append(sum_Dq_plus_lKLt_dot_KLt.item())
                        sum_Dq_plus_lKLt_dot_KLt *= lambda_sum
                        
                        if flag_iwae:
                            loss = 1 / x.size(0) * loss_iwae_gauss(mu, logvar, x, model.decode, k=sample_K0)
                        elif flag_vae:
                            loss = MSE + KL
                        else:
                            loss = MSE + sum_Dq_plus_lKLt_dot_KLt


                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()
                        losses.append(loss.item())

                    # 3. 记录每个 epoch 的平均 loss
                    mean_loss = np.mean(losses)
                    mean_MSE = np.mean(MSEs)
                    # mean_BCE = np.mean(BCEs)
                    mean_Dq = np.mean(Dqs)
                    mean_KL = np.mean(KLs)
                    mean_KL_t = np.mean(KL_ts)
                    mean_Dq_plus_lKLt_dot_KLt = np.mean(sum_Dq_plus_lKLt_dot_KLt_s)

                    writer.add_scalar("Loss/train_epoch", mean_loss, epoch)
                    writer.add_scalar("MSE/train_epoch", mean_MSE, epoch)
                    # writer.add_scalar("BCE/train_epoch", mean_BCE, epoch)
                    writer.add_scalar("Dq/train_epoch", mean_Dq, epoch)
                    writer.add_scalar("KL/train_epoch", mean_KL, epoch)
                    writer.add_scalar("KL_t/train_epoch", mean_KL_t, epoch)
                    writer.add_scalar("Sum/train_epoch", mean_Dq_plus_lKLt_dot_KLt, epoch)

                    write_test_images(writer, model, test_loader, device, epoch)
                    
                    fid_score = eval_fid(model, val_loader, device, fid_metric)
                    writer.add_scalar("FID/val_epoch", fid_score, epoch)
                    print(
                        f"[Epoch {epoch+1}] Loss: {mean_loss:.4f}, FID: {fid_score:.4f}, BCE: {'N/A'}"
                    )
                    print(
                        f"           MSE: {mean_MSE:.4f}, Dq: {mean_Dq:.4f}, KL: {mean_KL:.4f}, KL_t: {mean_KL_t:.4f}"
                    )
                    torch.cuda.empty_cache()
                    
                write_test_images(
                    writer,
                    model,
                    test_loader,
                    device,
                    global_step=0,
                    sample_K=42,
                    folder_name="img_final_epoch",
                )
                
                fid_score_final = eval_fid(model, test_loader, device, fid_metric)
                writer.add_scalar("FID/test_epoch", fid_score_final, epoch)
                print(f"Final Test FID: {fid_score_final:.4f}")

                hparam_dict = {
                    "batch_size": batch_size0,
                    "epochs": epoch0,
                    "lr": lr0,
                    "K": sample_K0,
                    "sigma2": sigma20,
                    "lambda_D": lambda_D0,
                    "lambda_KL": lambda_KL0,
                    "lambda_KLt": lambda_KLt,
                    "lambda_sum": lambda_sum,
                    "tensround": tensround,
                }
                metric_dict = {
                    "last loss": mean_loss,
                    "last FID": float(fid_score_final),
                    "last MSE": mean_MSE,
                }
                writer.add_hparams(hparam_dict, metric_dict)
                # 4. 训练结束后关闭 writer
                writer.close()
                
                torch.save(
                    model,
                    f"{model_dir}/final_model.pt",
                )
                

Using device: cuda:2


/home/yzm/anaconda3/envs/gen/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:36: UserWarning: Metric `FrechetInceptionDistance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


[Epoch 1] Loss: 988.7470, FID: 412.1866, BCE: N/A
           MSE: 193.0931, Dq: 1580.1851, KL: 29.3407, KL_t: 278.0708
[Epoch 2] Loss: 183.3701, FID: 415.4815, BCE: N/A
           MSE: 183.4706, Dq: -0.2372, KL: 3.9219, KL_t: 0.9042
[Epoch 3] Loss: 148.5870, FID: 414.3927, BCE: N/A
           MSE: 151.3311, Dq: -5.5287, KL: 6.8355, KL_t: 1.0145
[Epoch 4] Loss: 128.6003, FID: 405.6848, BCE: N/A
           MSE: 133.2738, Dq: -9.4266, KL: 7.9735, KL_t: 1.9921
[Epoch 5] Loss: 108.9850, FID: 375.0819, BCE: N/A
           MSE: 113.6682, Dq: -9.4609, KL: 11.9913, KL_t: 2.3596
[Epoch 6] Loss: 99.4451, FID: 379.6181, BCE: N/A
           MSE: 104.4262, Dq: -10.0646, KL: 14.1959, KL_t: 2.5597
[Epoch 7] Loss: 89.8846, FID: 366.2059, BCE: N/A
           MSE: 95.1950, Dq: -10.7246, KL: 16.4505, KL_t: 2.5974
[Epoch 8] Loss: 84.3949, FID: 349.4412, BCE: N/A
           MSE: 89.7863, Dq: -10.9067, KL: 17.7404, KL_t: 3.0939
[Epoch 9] Loss: 79.2759, FID: 342.6859, BCE: N/A
           MSE: 84.6609, Dq: -10

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-6_5iyypn'


[Epoch 10] Loss: 75.7251, FID: 338.2493, BCE: N/A
           MSE: 81.0526, Dq: -10.8903, KL: 20.6173, KL_t: 5.8831
[Epoch 11] Loss: 72.5650, FID: 321.0711, BCE: N/A
           MSE: 77.9497, Dq: -10.9584, KL: 21.5821, KL_t: 4.7262
[Epoch 12] Loss: 70.4489, FID: 314.6226, BCE: N/A
           MSE: 75.7561, Dq: -10.8880, KL: 22.5992, KL_t: 6.8405
[Epoch 13] Loss: 68.5259, FID: 312.6651, BCE: N/A
           MSE: 73.8306, Dq: -10.8338, KL: 23.5585, KL_t: 5.6087
[Epoch 14] Loss: 66.9177, FID: 289.0131, BCE: N/A
           MSE: 71.8971, Dq: -10.3865, KL: 24.8751, KL_t: 10.6911
[Epoch 15] Loss: 65.5649, FID: 287.5641, BCE: N/A
           MSE: 70.5035, Dq: -10.2441, KL: 25.6500, KL_t: 9.1698
[Epoch 16] Loss: 64.3406, FID: 281.8836, BCE: N/A
           MSE: 69.1061, Dq: -9.8868, KL: 26.7522, KL_t: 8.8977
[Epoch 17] Loss: 62.7530, FID: 270.3133, BCE: N/A
           MSE: 67.3554, Dq: -9.6107, KL: 27.8751, KL_t: 10.1462
[Epoch 18] Loss: 61.4220, FID: 265.9539, BCE: N/A
           MSE: 65.6446, Dq: -

...

[Epoch 34] Loss: 49.8129, FID: 226.8626, BCE: N/A
           MSE: 51.4018, Dq: -3.4504, KL: 46.4151, KL_t: 6.8122
[Epoch 35] Loss: 49.5683, FID: 224.1040, BCE: N/A
           MSE: 51.1295, Dq: -3.3567, KL: 46.8280, KL_t: 5.8564
[Epoch 36] Loss: 49.8540, FID: 230.0051, BCE: N/A
           MSE: 51.4188, Dq: -3.4216, KL: 46.4836, KL_t: 7.2989
[Epoch 37] Loss: 49.6069, FID: 224.9143, BCE: N/A
           MSE: 51.1850, Dq: -3.4122, KL: 46.7137, KL_t: 6.3959
[Epoch 38] Loss: 49.7059, FID: 225.5706, BCE: N/A
           MSE: 51.3146, Dq: -3.5002, KL: 46.3000, KL_t: 7.0679
[Epoch 39] Loss: 49.4254, FID: 228.9219, BCE: N/A
           MSE: 51.0622, Dq: -3.5193, KL: 46.2992, KL_t: 6.1406
[Epoch 40] Loss: 49.5829, FID: 225.4852, BCE: N/A
           MSE: 51.1788, Dq: -3.4615, KL: 46.4615, KL_t: 6.7399
[Epoch 41] Loss: 49.5719, FID: 225.7875, BCE: N/A
           MSE: 51.1706, Dq: -3.4577, KL: 46.4242, KL_t: 6.5067
[Epoch 42] Loss: 49.5020, FID: 224.8074, BCE: N/A
           MSE: 51.0844, Dq: -3.4269, 

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-shijmqlj'


[Epoch 44] Loss: 49.6059, FID: 228.8335, BCE: N/A
           MSE: 50.4022, Dq: -1.8756, KL: 72.4229, KL_t: 7.0737
[Epoch 45] Loss: 49.9291, FID: 230.4691, BCE: N/A
           MSE: 50.6936, Dq: -1.8789, KL: 72.1709, KL_t: 8.7448
[Epoch 46] Loss: 49.4827, FID: 226.9875, BCE: N/A
           MSE: 50.3294, Dq: -1.9549, KL: 72.0256, KL_t: 6.5368
[Epoch 47] Loss: 49.6027, FID: 232.2761, BCE: N/A
           MSE: 50.4064, Dq: -1.8996, KL: 72.1529, KL_t: 7.3086


Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-vorqj3im'


[Epoch 48] Loss: 49.4962, FID: 227.8882, BCE: N/A
           MSE: 50.3084, Dq: -1.8974, KL: 72.2205, KL_t: 6.8228
[Epoch 49] Loss: 49.5082, FID: 227.8149, BCE: N/A
           MSE: 50.2982, Dq: -1.8684, KL: 72.3965, KL_t: 7.2075
[Epoch 50] Loss: 49.4153, FID: 227.1745, BCE: N/A
           MSE: 50.1916, Dq: -1.8310, KL: 72.6829, KL_t: 6.9588
[Epoch 51] Loss: 49.5636, FID: 230.4654, BCE: N/A
           MSE: 50.3364, Dq: -1.8671, KL: 72.3631, KL_t: 8.0346
[Epoch 52] Loss: 49.3397, FID: 226.4207, BCE: N/A
           MSE: 50.1390, Dq: -1.8875, KL: 72.3053, KL_t: 7.2182
[Epoch 53] Loss: 49.4470, FID: 225.8382, BCE: N/A
           MSE: 50.2125, Dq: -1.8374, KL: 72.5840, KL_t: 7.6594
[Epoch 54] Loss: 49.3440, FID: 226.1118, BCE: N/A
           MSE: 50.1104, Dq: -1.8343, KL: 72.7706, KL_t: 7.5357
[Epoch 55] Loss: 49.2279, FID: 223.6439, BCE: N/A
           MSE: 49.9747, Dq: -1.7657, KL: 73.1857, KL_t: 6.8051
[Epoch 56] Loss: 49.1814, FID: 225.0135, BCE: N/A
           MSE: 49.9563, Dq: -1.8226, 

In [19]:
#### ours
batch_size0 = 128
batch_size_test = 64
batch_size_test_nll = 16
epoch0 = 100
lr0 = 1e-4


sigma20 = 1
prior_var0 = 1
lambda_t0 = 0.1
lambda_KLt = 0.1
lambda_D0 = 0.5
lambda_KL0 = 0
lambda_sum = 0.5 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1}
lambda_KL0 = 0 # vae的 KL，非vae时=0

torch.cuda.empty_cache()

device = torch.device(f"cuda:{cuda_num}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

flag_vae = True
flag_iwae = False
flag_cv = False
prefix = f"F3-add-latent{LATENT_DIM}"
folder = "runs__re"
file_idx = 4444
sample_K0 = 2

lambda_D0 = 0.5
lambda_sum = 1 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1} 的分母
num_KLt = 1.0 # BaM 的 KL{t+1} 的分子

sigma20 = 1
prior_var0 = 1
sigma2_list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
prior_var_list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
sigma2_list = [0.01, 1.0, 10.0]
prior_var_list = [0.01, 1.0, 10.0]
for lr0 in [5e-4]:
    for sigma20 in [1]:
        for prior_var0 in prior_var_list:    
            for tensround in range(file_idx, file_idx + 1):
                time_now = datetime.now().strftime("%m-%d_%H-%M")
                if flag_vae:
                    lambda_sum = 0
                    lambda_KLt = 1
                    lambda_KL0 = 1
                    model_dir=f"{folder}/{prefix}-{datasetname}-vae-s2lk{sigma20}-s2p{prior_var0}-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir 
                    )
                elif flag_iwae:
                    model_dir=f"{folder}/{prefix}-{datasetname}-iwae-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                elif flag_cv:
                    model_dir=f"{folder}/{prefix}-{datasetname}-cv-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                else:
                    model_dir=f"{folder}/{prefix}-{datasetname}-ours-s2lk{sigma20}-s2p{prior_var0}-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{num_KLt/lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                    log_dir=model_dir
                )
                model = VAE().to(device)
                model_t = VAE().to(device)
                optimizer = torch.optim.Adam(model.parameters(), lr=lr0)

                fid_metric = FrechetInceptionDistance(feature=2048).to(device)

                data_loader = DataLoader(
                    train_dataset_torch, batch_size=batch_size0, shuffle=True,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                val_loader = DataLoader(
                    val_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4, worker_init_fn=seed_worker, generator=gen
                )
                                

                test_loader = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                test_loader_nll = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test_nll, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )

                plt_ck_p = epoch0 // 20  # 显示 20 次测试结果

                total_steps = epoch0 * len(data_loader)

                for epoch in range(epoch0):

                    losses = []
                    Dqs = []
                    KLs = []
                    KL_ts = []
                    MSEs = []
                    BCEs = []
                    sum_Dq_plus_lKLt_dot_KLt_s = []

                    alpha_max = 1.0
                    beta_max = 1.0
                    gamma_max = 1.0

                    warmup1 = 30      # 只用 MSE+KL
                    warmup2 = 80      # 完整过渡到 MSE+KL+BaM
                    # alpha = alpha_max * min(1.0, epoch / warmup1)
                    alpha = alpha_max
                    gamma = gamma_max
                    beta = beta_max
                    
                    old_enc_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.encoder.state_dict().items()
                    }

                    for batch_idx, (x, y) in enumerate(data_loader):
                        x = x.to(device)  # [batch, ...]
                        recon_batch, mu, logvar = model(x)  # [batch, D]

                        # 恢复旧 encoder 的参数到临时 old_encoder 推断（在 CPU 拷贝上不需要梯度）
                        with torch.no_grad():
                            # 将保存的 old state 恢复到 model.encoder 的一个临时副本计算 mu_old/logvar_old
                            # 更高效：直接在当前 encoder 上 load_state_dict(old_enc_state)，但要小心线程安全与 GPU；这里用简单方式：
                            # 创建新 encoder 实例会更健壮但也更重 —— 下面采用直接把旧参数载入 model.encoder 并再恢复当前参数的办法（较快）
                            model_t.encoder.load_state_dict(old_enc_state)   # load old weights
                            # 注意：mu_old, logvar_old 都是在 no_grad 环境下计算的（无需保持 grad）
                            mu_old, logvar_old = model_t.encoder(x)         # compute old outputs

                        global_step = epoch * len(data_loader) + batch_idx

                        KL = 1.0 / x.size(0) * loss_function(x, recon_batch, mu, logvar, sigma2=sigma20, prior_var=prior_var0)
                        KLs.append(KL.item())
                        
                        if not flag_vae and not flag_iwae:                            
                            if flag_cv:
                                Dq = cv_bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K1=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    writer=writer, global_step=global_step)
                            else:
                                Dq = bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    prior_var=prior_var0)
                        else:
                            Dq = torch.tensor(0.0)
                        Dqs.append(Dq.item())
                        
                        MSE = nn.functional.mse_loss(
                            recon_batch, x, reduction="sum"
                        ) / x.size(0)

                        MSEs.append(MSE.item())
                        MSE = (1.0 / sigma20) * MSE

                        var = torch.exp(logvar)
                        var_old = torch.exp(logvar_old)
                        # 先在 latent 维度 D 上求和
                        KL_t_per_latent = 0.5 * (
                            torch.log(var / var_old) +
                            (var_old + (mu_old - mu).pow(2)) / var - 1
                        ).sum(dim=1)                        # [N]
                        # 再对 batch 维度求均值
                        KL_t = KL_t_per_latent.mean()        # scalar

                        KL_ts.append(KL_t.item())

                        Dq *= lambda_D0
                        KL *= lambda_KL0
                        KL_t *= num_KLt / lambda_KLt

                        sum_Dq_plus_lKLt_dot_KLt = Dq + KL_t
                        sum_Dq_plus_lKLt_dot_KLt_s.append(sum_Dq_plus_lKLt_dot_KLt.item())
                        sum_Dq_plus_lKLt_dot_KLt *= lambda_sum
                        
                        if flag_iwae:
                            loss = 1 / x.size(0) * loss_iwae_gauss(mu, logvar, x, model.decode, k=sample_K0)
                        elif flag_vae:
                            loss = MSE + KL
                        else:
                            loss = MSE + sum_Dq_plus_lKLt_dot_KLt


                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()
                        losses.append(loss.item())

                    # 3. 记录每个 epoch 的平均 loss
                    mean_loss = np.mean(losses)
                    mean_MSE = np.mean(MSEs)
                    # mean_BCE = np.mean(BCEs)
                    mean_Dq = np.mean(Dqs)
                    mean_KL = np.mean(KLs)
                    mean_KL_t = np.mean(KL_ts)
                    mean_Dq_plus_lKLt_dot_KLt = np.mean(sum_Dq_plus_lKLt_dot_KLt_s)

                    writer.add_scalar("Loss/train_epoch", mean_loss, epoch)
                    writer.add_scalar("MSE/train_epoch", mean_MSE, epoch)
                    # writer.add_scalar("BCE/train_epoch", mean_BCE, epoch)
                    writer.add_scalar("Dq/train_epoch", mean_Dq, epoch)
                    writer.add_scalar("KL/train_epoch", mean_KL, epoch)
                    writer.add_scalar("KL_t/train_epoch", mean_KL_t, epoch)
                    writer.add_scalar("Sum/train_epoch", mean_Dq_plus_lKLt_dot_KLt, epoch)

                    write_test_images(writer, model, test_loader, device, epoch)
                    
                    fid_score = eval_fid(model, val_loader, device, fid_metric)
                    writer.add_scalar("FID/val_epoch", fid_score, epoch)
                    print(
                        f"[Epoch {epoch+1}] Loss: {mean_loss:.4f}, FID: {fid_score:.4f}, BCE: {'N/A'}"
                    )
                    print(
                        f"           MSE: {mean_MSE:.4f}, Dq: {mean_Dq:.4f}, KL: {mean_KL:.4f}, KL_t: {mean_KL_t:.4f}"
                    )
                    torch.cuda.empty_cache()
                    
                write_test_images(
                    writer,
                    model,
                    test_loader,
                    device,
                    global_step=0,
                    sample_K=42,
                    folder_name="img_final_epoch",
                )
                
                fid_score_final = eval_fid(model, test_loader, device, fid_metric)
                writer.add_scalar("FID/test_epoch", fid_score_final, epoch)
                print(f"Final Test FID: {fid_score_final:.4f}")

                hparam_dict = {
                    "batch_size": batch_size0,
                    "epochs": epoch0,
                    "lr": lr0,
                    "K": sample_K0,
                    "sigma2": sigma20,
                    "lambda_D": lambda_D0,
                    "lambda_KL": lambda_KL0,
                    "lambda_KLt": lambda_KLt,
                    "lambda_sum": lambda_sum,
                    "tensround": tensround,
                }
                metric_dict = {
                    "last loss": mean_loss,
                    "last FID": float(fid_score_final),
                    "last MSE": mean_MSE,
                }
                writer.add_hparams(hparam_dict, metric_dict)
                # 4. 训练结束后关闭 writer
                writer.close()
                
                torch.save(
                    model,
                    f"{model_dir}/final_model.pt",
                )
                

Using device: cuda:2


/home/yzm/anaconda3/envs/gen/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:36: UserWarning: Metric `FrechetInceptionDistance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


[Epoch 1] Loss: 172.9199, FID: 383.9702, BCE: N/A
           MSE: 150.1355, Dq: 0.0000, KL: 22.7844, KL_t: 1024.9313
[Epoch 2] Loss: 120.0428, FID: 360.4411, BCE: N/A
           MSE: 109.4954, Dq: 0.0000, KL: 10.5474, KL_t: 3.1170
[Epoch 3] Loss: 109.6198, FID: 340.3269, BCE: N/A
           MSE: 96.3898, Dq: 0.0000, KL: 13.2299, KL_t: 3.4246
[Epoch 4] Loss: 102.4144, FID: 313.0909, BCE: N/A
           MSE: 87.8562, Dq: 0.0000, KL: 14.5583, KL_t: 3.2157
[Epoch 5] Loss: 97.7799, FID: 294.8948, BCE: N/A
           MSE: 82.5377, Dq: 0.0000, KL: 15.2422, KL_t: 3.0752
[Epoch 6] Loss: 94.4321, FID: 291.3057, BCE: N/A
           MSE: 79.1011, Dq: 0.0000, KL: 15.3310, KL_t: 3.5526
[Epoch 7] Loss: 91.6065, FID: 285.9906, BCE: N/A
           MSE: 76.2808, Dq: 0.0000, KL: 15.3257, KL_t: 3.6588
[Epoch 8] Loss: 89.8686, FID: 286.3603, BCE: N/A
           MSE: 74.4680, Dq: 0.0000, KL: 15.4006, KL_t: 2.6122


Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-a9bmbdkj'


[Epoch 9] Loss: 88.8175, FID: 279.9440, BCE: N/A
           MSE: 73.4898, Dq: 0.0000, KL: 15.3277, KL_t: 3.3710
[Epoch 10] Loss: 87.3071, FID: 276.0470, BCE: N/A
           MSE: 71.8005, Dq: 0.0000, KL: 15.5066, KL_t: 3.0920
[Epoch 11] Loss: 86.9922, FID: 268.3818, BCE: N/A
           MSE: 71.3439, Dq: 0.0000, KL: 15.6483, KL_t: 3.2527
[Epoch 12] Loss: 86.6974, FID: 263.2596, BCE: N/A
           MSE: 70.8065, Dq: 0.0000, KL: 15.8909, KL_t: 7.0950
[Epoch 13] Loss: 85.7233, FID: 261.0984, BCE: N/A
           MSE: 69.4648, Dq: 0.0000, KL: 16.2585, KL_t: 4.1377
[Epoch 14] Loss: 85.3158, FID: 261.4742, BCE: N/A
           MSE: 68.7346, Dq: 0.0000, KL: 16.5812, KL_t: 3.9557
[Epoch 15] Loss: 85.4937, FID: 259.7333, BCE: N/A
           MSE: 68.7218, Dq: 0.0000, KL: 16.7719, KL_t: 3.2347
[Epoch 16] Loss: 84.6931, FID: 249.3147, BCE: N/A
           MSE: 67.6748, Dq: 0.0000, KL: 17.0182, KL_t: 3.9823


...

[Epoch 11] Loss: 83.4566, FID: 278.3725, BCE: N/A
           MSE: 65.5048, Dq: 0.0000, KL: 17.9518, KL_t: 2.1111
[Epoch 12] Loss: 83.1842, FID: 269.5367, BCE: N/A
           MSE: 64.8476, Dq: 0.0000, KL: 18.3366, KL_t: 2.3946
[Epoch 13] Loss: 82.6782, FID: 261.1157, BCE: N/A
           MSE: 64.0067, Dq: 0.0000, KL: 18.6716, KL_t: 3.5487
[Epoch 14] Loss: 82.1821, FID: 267.1701, BCE: N/A
           MSE: 63.4610, Dq: 0.0000, KL: 18.7211, KL_t: 2.7601
[Epoch 15] Loss: 81.9860, FID: 258.1570, BCE: N/A
           MSE: 63.2383, Dq: 0.0000, KL: 18.7477, KL_t: 3.7958
[Epoch 16] Loss: 82.6318, FID: 257.7420, BCE: N/A
           MSE: 63.8917, Dq: 0.0000, KL: 18.7401, KL_t: 2.9356
[Epoch 17] Loss: 82.0117, FID: 257.3065, BCE: N/A
           MSE: 63.3120, Dq: 0.0000, KL: 18.6997, KL_t: 2.1616
[Epoch 18] Loss: 81.9088, FID: 258.2316, BCE: N/A
           MSE: 63.2224, Dq: 0.0000, KL: 18.6865, KL_t: 2.4499
[Epoch 19] Loss: 82.0160, FID: 254.8328, BCE: N/A
           MSE: 63.3758, Dq: 0.0000, KL: 18.64

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-q17gjyvj'


[Epoch 79] Loss: 79.4951, FID: 230.3040, BCE: N/A
           MSE: 60.0080, Dq: 0.0000, KL: 19.4870, KL_t: 2.0133
[Epoch 80] Loss: 79.5758, FID: 228.3175, BCE: N/A
           MSE: 60.1060, Dq: 0.0000, KL: 19.4698, KL_t: 2.0634
[Epoch 81] Loss: 79.4324, FID: 227.9811, BCE: N/A
           MSE: 59.9315, Dq: 0.0000, KL: 19.5009, KL_t: 1.9241
[Epoch 82] Loss: 79.3630, FID: 231.7763, BCE: N/A
           MSE: 59.8635, Dq: 0.0000, KL: 19.4995, KL_t: 2.0588
[Epoch 83] Loss: 79.6332, FID: 229.7313, BCE: N/A
           MSE: 60.1777, Dq: 0.0000, KL: 19.4555, KL_t: 3.1515
[Epoch 84] Loss: 79.3892, FID: 227.6142, BCE: N/A
           MSE: 59.8800, Dq: 0.0000, KL: 19.5092, KL_t: 1.8259
[Epoch 85] Loss: 79.4566, FID: 227.7379, BCE: N/A
           MSE: 59.9661, Dq: 0.0000, KL: 19.4905, KL_t: 2.3428
[Epoch 86] Loss: 79.3033, FID: 232.1751, BCE: N/A
           MSE: 59.8070, Dq: 0.0000, KL: 19.4963, KL_t: 1.5761
[Epoch 87] Loss: 79.3513, FID: 227.9357, BCE: N/A
           MSE: 59.8460, Dq: 0.0000, KL: 19.50

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-x5wrp_j3'


[Epoch 24] Loss: 82.9923, FID: 254.2628, BCE: N/A
           MSE: 65.9357, Dq: 0.0000, KL: 17.0565, KL_t: 2.5370
[Epoch 25] Loss: 82.5631, FID: 248.3683, BCE: N/A
           MSE: 64.9824, Dq: 0.0000, KL: 17.5807, KL_t: 6.6145
[Epoch 26] Loss: 82.5660, FID: 247.3679, BCE: N/A
           MSE: 64.6979, Dq: 0.0000, KL: 17.8681, KL_t: 1.9166
[Epoch 27] Loss: 82.0743, FID: 249.0890, BCE: N/A
           MSE: 64.1258, Dq: 0.0000, KL: 17.9484, KL_t: 1.7662
[Epoch 28] Loss: 81.9188, FID: 247.5220, BCE: N/A
           MSE: 63.9275, Dq: 0.0000, KL: 17.9913, KL_t: 2.6726
[Epoch 29] Loss: 81.9290, FID: 243.1256, BCE: N/A
           MSE: 63.8954, Dq: 0.0000, KL: 18.0336, KL_t: 3.1622
[Epoch 30] Loss: 81.8646, FID: 248.8750, BCE: N/A
           MSE: 63.7973, Dq: 0.0000, KL: 18.0672, KL_t: 1.7808
[Epoch 31] Loss: 81.5212, FID: 244.5040, BCE: N/A
           MSE: 63.4102, Dq: 0.0000, KL: 18.1110, KL_t: 3.3890
[Epoch 32] Loss: 81.6696, FID: 246.8294, BCE: N/A
           MSE: 63.5603, Dq: 0.0000, KL: 18.10

## eof